# Value of Information Analysis: van Zwet (2026) OSC Corpus

This notebook computes Bayesian decision-theoretic **value of information (VOI)** quantities
for two latent epistemic states and three signals, using the signal-to-noise ratio mixture
model of van Zwet, Gelman & Więcek (2026) fit to the **Open Science Collaboration (OSC)**
psychology replication corpus.

## Decision problem

A reviewer must choose between two actions: **flag** a claim for evidential fragility, or
**do not flag** it. We use a symmetric 0/1 utility: the reviewer earns 1 for a correct
decision and 0 for an incorrect one. Expected utility therefore equals **probability of
correct decision**.

We evaluate two candidate latent epistemic states $\\theta$:

- $\\theta_\\mathrm{snr} = \\mathbf{1}(|\\Lambda| \\ge \\lambda_0)$: the study has
  signal-to-noise ratio at least $\\lambda_0 = 2.0$
- $\\theta_\\mathrm{sign} = \\mathbf{1}(\\Lambda Z > 0)$: the observed effect has the
  correct sign

and three observable signals:

- $S_Z = Z$ — the original study's continuous $z$-value
- $S_\\mathrm{sig} = \\mathbf{1}(|Z| \\ge 1.96)$ — statistical significance
- $S_\\mathrm{rep} = \\mathbf{1}(Z Z_\\mathrm{rep} > 0,\ |Z_\\mathrm{rep}| \\ge 1.96)$ — exact
  replication success

## VOI quantities

For a signal $S$, the benchmark value is
$$V(\\theta; S) = \\mathbb{E}_S\\bigl[\\max\\{P(\\theta=1\\mid S),\\, P(\\theta=0\\mid S)\\}\\bigr]$$
and the value of information is $\\Delta(\\theta; S) = V(\\theta; S) - V_0(\\theta)$, where
$V_0(\\theta) = \\max\\{P(\\theta=1), P(\\theta=0)\\}$ is the no-signal baseline.

## Signal-to-noise model

The latent SNR $\\Lambda \\sim H$ (a 4-component symmetric mixture of normals). The observed
$z$-value is $Z = \\Lambda + \\varepsilon$, $\\varepsilon \\sim \\mathcal{N}(0,1)$. An exact
replication produces $Z_\\mathrm{rep} = \\Lambda + \\varepsilon'$, independent of $Z$ given
$\\Lambda$.

## OSC corpus parameters

Fitted by maximum likelihood from absolute $z$-values of 100 OSC replication studies.
Parameters read from the [BEAR GitHub repository](https://github.com/wwiecek/BEAR/) `.rds` files.

In [ ]:
import numpy as np
from scipy import stats
import pandas as pd

## 1. OSC Corpus Parameters

The prior on $\\Lambda$ is a symmetric 4-component mixture of normals
$H = \\sum_k p_k \\, \\mathcal{N}(0, \\sigma_{\\mathrm{SNR},k}^2)$.

Observed $z$-values are distributed as $\\sum_k p_k \\, \\mathcal{N}(0, \\sigma_{z,k}^2)$
where $\\sigma_{z,k}^2 = \\sigma_{\\mathrm{SNR},k}^2 + 1$.

Component 4 ($\\sigma_{\\mathrm{SNR}} \\approx 2252$) represents a tiny fraction of studies
with extremely large effects; almost all of its probability mass falls outside any
finite grid and is handled analytically (see Section 4).

In [ ]:
# OSC corpus: 4-component mixture parameters from van Zwet et al. (2026) / BEAR repo
p_mix  = np.array([0.390982, 0.112415, 0.486510, 0.010093])   # mixture weights
ssnr   = np.array([0.446322, 8.251094, 2.436323, 2251.593835]) # sigma_SNR per component
sz_mix = np.array([1.095082, 8.311471, 2.633566, 2251.594057]) # sigma_z  per component

# SNR threshold for theta_snr
lam0 = 2.0

print("Component weights and sigmas:")
print(f"{'k':>3}  {'p_k':>9}  {'sigma_SNR':>12}  {'sigma_z':>12}")
for k in range(4):
    print(f"{k+1:>3}  {p_mix[k]:>9.6f}  {ssnr[k]:>12.6f}  {sz_mix[k]:>12.6f}")

## 2. Integration Grids

We approximate the integrals over $\\lambda$ and $z$ using fine uniform grids covering
$[-30, 30]$. This range captures $> 99.97\\%$ of the marginal probability for components
1–3; component 4 ($\\sigma_z \\approx 2252$) is handled separately in Section 4.

The prior density on $\\lambda$ is computed on the grid, then renormalized to account
for the truncation to $[-30, 30]$. The chunk-based loop processes 200 $z$-values at a
time to keep memory usage manageable.

In [ ]:
# Grids: 2001 lambda points, 4001 z points, both over [-30, 30]
lam_grid = np.linspace(-30, 30, 2001);  dl = lam_grid[1] - lam_grid[0]
z_grid   = np.linspace(-30, 30, 4001);  dz = z_grid[1]   - z_grid[0]

# Prior density h(lambda) = sum_k p_k * N(lambda; 0, sigma_SNR_k^2)
# (renormalized after truncating to the grid)
h = np.sum(p_mix[:,None] * stats.norm.pdf(lam_grid[None,:], 0, ssnr[:,None]), axis=0)
h /= h.sum() * dl   # normalize so integral over grid = 1

# Marginal density of Z: f(z) = sum_k p_k * N(z; 0, sigma_z_k^2)
# (renormalized after truncating to the grid)
fz = np.sum(p_mix[:,None] * stats.norm.pdf(z_grid[None,:], 0, sz_mix[:,None]), axis=0)
fz /= fz.sum() * dz

# Indicator: |lambda| >= lambda_0  (used for theta_snr)
hs_lam = (np.abs(lam_grid) >= lam0).astype(float)

print(f"Grid spacings: dlambda={dl:.4f}, dz={dz:.4f}")
print(f"Prior integrates to: {(h * dl).sum():.6f} (should be 1.0 after renorm)")

## 3. Grid-Based Posterior Computation

For each $z$ value on the grid we compute:

1. **Posterior** $p(\\lambda \\mid z) \\propto \\phi(z - \\lambda)\\, h(\\lambda)$
2. **Replication likelihood** $L_1(\\lambda, z) = P(S_\\mathrm{rep}=1 \\mid \\lambda, z) = \\Phi(\\mathrm{sign}(z)\\cdot\\lambda - 1.96)$
3. **Posteriors given $S_\\mathrm{rep}=1$ or $0$**: $p(\\lambda \\mid z, s) \\propto p(\\lambda \\mid z) \\cdot L_1^s(1-L_1)^{1-s}$
4. **Posterior probabilities** of $\\theta_\\mathrm{sign}=1$ and $\\theta_\\mathrm{snr}=1$ under each conditioning set

Results are accumulated across all $z$ values for use in the VOI integrals.

In [ ]:
# Arrays to accumulate, indexed over z_grid
P_sign_z  = np.zeros(len(z_grid))  # P(theta_sign=1 | z)
P_snr_z   = np.zeros(len(z_grid))  # P(theta_snr=1  | z)
P_srep1_z = np.zeros(len(z_grid))  # P(S_rep=1 | z)

# Posterior theta probabilities given (z, S_rep=1) and (z, S_rep=0)
Ps_z1 = np.zeros(len(z_grid))   # P(theta_sign=1 | z, S_rep=1)
Ps_z0 = np.zeros(len(z_grid))   # P(theta_sign=1 | z, S_rep=0)
Pn_z1 = np.zeros(len(z_grid))   # P(theta_snr=1  | z, S_rep=1)
Pn_z0 = np.zeros(len(z_grid))   # P(theta_snr=1  | z, S_rep=0)

CHUNK = 200  # process this many z-values at a time to limit peak memory

for start in range(0, len(z_grid), CHUNK):
    zc = z_grid[start:start+CHUNK]     # (nc,)
    nc = len(zc)
    zs = np.sign(zc); zs[zs==0] = 1.0  # sign of z, for replication likelihood

    # --- posterior p(lambda | z) for each z in chunk ---
    # lik[i,j] = phi(z_i - lambda_j): likelihood of observing z_i given lambda_j
    lik  = stats.norm.pdf(zc[:,None] - lam_grid[None,:])   # (nc, n_lam)
    post = h[None,:] * lik                                  # unnormalized posterior
    norm = post.sum(axis=1, keepdims=True) * dl
    post /= norm                                            # normalized posterior

    # --- replication success likelihood ---
    # P(S_rep=1 | lambda, z) = Phi(sign(z)*lambda - 1.96)
    # This is the probability the replication z-value has the same sign as z
    # and exceeds 1.96 in magnitude
    L1   = stats.norm.cdf(zs[:,None] * lam_grid[None,:] - 1.96)  # (nc, n_lam)

    # Marginal P(S_rep=1|z) and P(S_rep=0|z)
    raw1 = post * L1;         ps1 = raw1.sum(axis=1) * dl   # (nc,)
    raw0 = post * (1 - L1);   ps0 = raw0.sum(axis=1) * dl

    # Posteriors given S_rep (normalized within each replication outcome)
    post1 = raw1 / np.maximum(ps1[:,None], 1e-300)   # p(lambda | z, S_rep=1)
    post0 = raw0 / np.maximum(ps0[:,None], 1e-300)   # p(lambda | z, S_rep=0)

    # --- correct-sign indicator: lambda * sign(z) > 0  <=>  theta_sign = 1 ---
    cs_mat = (zs[:,None] * lam_grid[None,:] > 0).astype(float)  # (nc, n_lam)

    # Posterior probabilities of each theta state
    P_sign_z[start:start+nc]  = (post  * cs_mat).sum(1) * dl
    P_snr_z[start:start+nc]   = (post  * hs_lam[None,:]).sum(1) * dl
    P_srep1_z[start:start+nc] = ps1

    Ps_z1[start:start+nc] = (post1 * cs_mat).sum(1) * dl
    Ps_z0[start:start+nc] = (post0 * cs_mat).sum(1) * dl
    Pn_z1[start:start+nc] = (post1 * hs_lam[None,:]).sum(1) * dl
    Pn_z0[start:start+nc] = (post0 * hs_lam[None,:]).sum(1) * dl

print("Grid computation complete.")

## 4. Analytical Correction for Component 4

Component 4 has $\\sigma_z \\approx 2252$, so only about 1.1% of its $z$-values fall
within $[-30, 30]$. The remaining $\\approx 98.9\\%$ are outside our grid and must be
accounted for analytically.

For any $z$ value from component 4 with $|z| \\gg 1$:
- $\\lambda \\mid z, \\text{comp4} \\approx \\mathcal{N}(z, 1)$ (since $\\sigma_{\\mathrm{SNR}} \\gg 1$)
- $P(\\theta_\\mathrm{sign}=1 \\mid z, \\text{comp4}) = \\Phi(|z|) \\approx 1$
- $P(\\theta_\\mathrm{snr}=1 \\mid z, \\text{comp4}) = P(|\\mathcal{N}(z,1)| \\ge 2) \\approx 1$ for $|z| \\gg 2$

Therefore every outside-grid $z$ from component 4 contributes $\\max(P(\\theta=1|z), P(\\theta=0|z)) = 1$
to all signal-based $V$ quantities.

**Effect on $V_0$:**
- $\\theta_\\mathrm{sign}$: comp4 increases $P(\\theta_\\mathrm{sign}=1)$, so $V_0$ increases by $\\mathrm{d}V$
- $\\theta_\\mathrm{snr}$: comp4 also increases $P(\\theta_\\mathrm{snr}=1)$, but since this is
  already $< 0.5$ (best action is to flag), $V_0 = 1 - P(\\theta_\\mathrm{snr}=1)$ **decreases** by $\\mathrm{d}V$.

**Effect on all signal-based $V$ quantities:** each increases by $+\\mathrm{d}V$, so
**$\\Delta$ values for $\\theta_\\mathrm{sign}$ are unchanged**, while
**$\\Delta$ values for $\\theta_\\mathrm{snr}$ each increase by $\\approx 2\\,\\mathrm{d}V \\approx 0.020$**.

In [ ]:
# Fraction of comp4 z-values that fall inside [-30, 30]
sigma_z4    = sz_mix[3]                          # ≈ 2251.6
p4_inside   = 2 * stats.norm.cdf(30 / sigma_z4) - 1  # ≈ 0.0107
# Weight of comp4 probability mass outside the grid
dV          = p_mix[3] * (1 - p4_inside)         # ≈ 0.009986

print(f"Comp4 sigma_z = {sigma_z4:.1f}")
print(f"Fraction of comp4 z-values inside [-30, 30]: {p4_inside:.5f}")
print(f"Outside-grid correction dV = {dV:.6f}")

## 5. VOI Computation

We now integrate the posterior probabilities (accumulated in Section 3) over the
marginal distribution of $z$ (or the signal) to obtain the benchmark values.

In [ ]:
# Helper: integrate arr(z) * f(z) dz over the z-grid
def integrate(arr):
    return (arr * fz).sum() * dz

# ---------- Prior marginals (grid-only, before comp4 correction) ----------
P0s_grid = integrate(P_sign_z)   # P(theta_sign=1) from grid
P0n_grid = integrate(P_snr_z)    # P(theta_snr=1)  from grid

# Apply comp4 correction to priors
P0s = P0s_grid + dV   # comp4 outside-grid contributes P(theta_sign=1|comp4) ≈ 1
P0n = P0n_grid + dV   # comp4 outside-grid contributes P(theta_snr=1|comp4)  ≈ 1

# V0: best constant-action value
# For theta_sign: P0s > 0.5 → don't flag always → V0 = P0s
# For theta_snr:  P0n < 0.5 → always flag        → V0 = 1 - P0n
V0s = max(P0s, 1 - P0s)
V0n = max(P0n, 1 - P0n)

# ---------- V_Z: observe continuous z-value ----------
VZ_s = integrate(np.maximum(P_sign_z, 1 - P_sign_z)) + dV
VZ_n = integrate(np.maximum(P_snr_z,  1 - P_snr_z))  + dV

# ---------- Marginal signal probabilities ----------
p_rep1   = integrate(P_srep1_z) + dV   # P(S_rep=1);  comp4 almost always produces S_rep=1
p_rep0   = 1 - p_rep1
sig_mask = np.abs(z_grid) >= 1.96
p_sig1   = integrate(sig_mask.astype(float)) + dV  # P(S_sig=1); comp4 z-values are |z|>>1.96
p_sig0   = 1 - p_sig1

# ---------- V_sig: observe only significance indicator ----------
def v_sig(Pz):
    # Posterior P(theta=1|S_sig=s) = E[P(theta=1|z) | S_sig=s]
    pt1_sig = (integrate(Pz * sig_mask) + dV) / p_sig1   # sig=1 bin gets comp4
    pt0_sig =  integrate(Pz * ~sig_mask)      / p_sig0
    return p_sig1 * max(pt1_sig, 1 - pt1_sig) + p_sig0 * max(pt0_sig, 1 - pt0_sig)

Vsig_s = v_sig(P_sign_z)
Vsig_n = v_sig(P_snr_z)

# ---------- V_rep: observe only S_rep ----------
def v_rep(Pz1, Pz0):
    # Pz1[i] = P(theta=1 | z_i, S_rep=1);  Pz0[i] = P(theta=1 | z_i, S_rep=0)
    # E[P(theta=1|S_rep=1)] = E_z[Pz1 * P(S_rep=1|z)] / P(S_rep=1)
    pt1 = (integrate(Pz1 * P_srep1_z) + dV) / p_rep1
    pt0 =  integrate(Pz0 * (1 - P_srep1_z)) / p_rep0
    return p_rep1 * max(pt1, 1 - pt1) + p_rep0 * max(pt0, 1 - pt0)

Vrep_s = v_rep(Ps_z1, Ps_z0)
Vrep_n = v_rep(Pn_z1, Pn_z0)

# ---------- V_both: observe (Z, S_rep) jointly ----------
# For each z, integrate over S_rep outcomes weighted by P(S_rep|z)
Vboth_s = integrate(
    P_srep1_z * np.maximum(Ps_z1, 1-Ps_z1) +
    (1-P_srep1_z) * np.maximum(Ps_z0, 1-Ps_z0)
) + dV
Vboth_n = integrate(
    P_srep1_z * np.maximum(Pn_z1, 1-Pn_z1) +
    (1-P_srep1_z) * np.maximum(Pn_z0, 1-Pn_z0)
) + dV

# ---------- V_(sig, rep): observe (S_sig, S_rep) jointly ----------
def v_sig_rep(Pz1, Pz0):
    V = 0
    # Iterate over the four (S_sig, S_rep) combinations
    for sm, sm_label in [(~sig_mask, 'sig=0'), (sig_mask, 'sig=1')]:
        for Pzr, Pr_z, sr_label in [
            (Pz0, 1 - P_srep1_z, 'rep=0'),
            (Pz1, P_srep1_z,     'rep=1')
        ]:
            # Weight of this (sig, rep) cell; comp4 goes into (sig=1, rep=1)
            extra = dV if (sm is sig_mask and Pzr is Pz1) else 0.0
            denom = integrate(sm.astype(float) * Pr_z) + extra
            if denom < 1e-10:
                continue
            pt1 = (integrate(Pzr * sm * Pr_z) + extra) / denom
            V  += denom * max(pt1, 1 - pt1)
    return V

Vsrep_s = v_sig_rep(Ps_z1, Ps_z0)
Vsrep_n = v_sig_rep(Pn_z1, Pn_z0)

print("VOI computation complete.")

## 6. Results

In [ ]:
# Verification: compare computed marginals to van Zwet et al. (2026) reported values
print("=== Verification against van Zwet et al. (2026) reported statistics ===")
print(f"P(theta_sign=1) = {P0s:.4f}  [paper: 0.793]")
print(f"P(sig=1)        = {p_sig1:.4f}  [paper: 0.351]")
print(f"P(S_rep=1)      = {p_rep1:.4f}  [paper: 0.334]")
print(f"P(theta_snr=1)  = {P0n:.4f}")
print()

In [ ]:
# Collect all VOI quantities
rows = [
    ("V0",                V0s,              V0n),
    ("V_Z",               VZ_s,             VZ_n),
    ("Delta_Z",           VZ_s - V0s,       VZ_n - V0n),
    ("V_sig",             Vsig_s,           Vsig_n),
    ("Delta_sig",         Vsig_s - V0s,     Vsig_n - V0n),
    ("V_rep",             Vrep_s,           Vrep_n),
    ("Delta_rep",         Vrep_s - V0s,     Vrep_n - V0n),
    ("V_both",            Vboth_s,          Vboth_n),
    ("Delta_both",        Vboth_s - V0s,    Vboth_n - V0n),
    ("Delta_rep|Z",       Vboth_s - VZ_s,   Vboth_n - VZ_n),
    ("V_(sig,rep)",       Vsrep_s,          Vsrep_n),
    ("Delta_rep|sig",     Vsrep_s - Vsig_s, Vsrep_n - Vsig_n),
]

df = pd.DataFrame(rows, columns=["Quantity", "theta_sign", "theta_snr"])
df = df.set_index("Quantity")
print(f"OSC corpus | lambda_0 = {lam0}\n")
print(df.to_string(float_format="{:.4f}".format))

In [ ]:
# Monotonicity checks: more information should never hurt
checks = [
    ("V_Z >= V0      (sign)",       VZ_s  - V0s),
    ("V_Z >= V0      (snr)",        VZ_n  - V0n),
    ("V_both >= V_Z  (sign)",       Vboth_s - VZ_s),
    ("V_both >= V_Z  (snr)",        Vboth_n - VZ_n),
    ("V_both >= V_rep (sign)",      Vboth_s - Vrep_s),
    ("V_both >= V_rep (snr)",       Vboth_n - Vrep_n),
    ("V_(sig,rep) >= V_sig (snr)",  Vsrep_n - Vsig_n),
    ("V_Z >= V_sig   (snr)",        VZ_n - Vsig_n),
]
print("=== Monotonicity checks ===")
all_pass = True
for label, diff in checks:
    status = "OK" if diff >= -1e-4 else "FAIL"
    if status == "FAIL": all_pass = False
    print(f"  {label}: {diff:+.5f}  {status}")
print(f"\nAll checks passed: {all_pass}")